# Exp 7: Uber Fare Prediction (Linear Regression & Random Forest)
**Dataset:** `uber.csv` from https://www.kaggle.com/datasets/yasserh/uber-fares-dataset  (keep it in the same folder as this notebook)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Pre-process the dataset

In [ ]:
df = pd.read_csv("uber.csv")
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# drop unnecessary columns
df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")

# missing values
print(df.isnull().sum())
df = df.dropna()

# convert pickup_datetime to datetime and extract features
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df["year"] = df["pickup_datetime"].dt.year
df["dayofweek"] = df["pickup_datetime"].dt.dayofweek
df = df.drop(columns=["pickup_datetime"])

In [ ]:
# Haversine formula: distance (km) between pickup and drop-off points
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df["pickup_longitude"], df["pickup_latitude"],
                              df["dropoff_longitude"], df["dropoff_latitude"])
print(df[["fare_amount", "distance_km"]].head())

## 2. Identify outliers

In [ ]:
cols = ["fare_amount", "distance_km", "passenger_count"]
plt.figure(figsize=(15, 4))
for i, c in enumerate(cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(y=df[c])
    plt.title("Boxplot of " + c)
plt.show()

In [ ]:
# Remove invalid / outlier values
print("Rows before:", len(df))
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 100)]
df = df[(df["distance_km"] > 0) & (df["distance_km"] < 100)]
df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 6)]
df = df[(df["pickup_latitude"].between(-90, 90)) & (df["dropoff_latitude"].between(-90, 90))]
df = df[(df["pickup_longitude"].between(-180, 180)) & (df["dropoff_longitude"].between(-180, 180))]
print("Rows after :", len(df))

## 3. Check the correlation

In [ ]:
corr = df.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.show()
print(corr["fare_amount"].sort_values(ascending=False))

## 4. Implement Linear Regression and Random Forest Regression

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

## 5. Evaluate the models and compare R2, RMSE

In [ ]:
def evaluate(name, y_true, y_pred):
    return {"Model": name,
            "R2": r2_score(y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mean_squared_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}

results = pd.DataFrame([evaluate("Linear Regression", y_test, y_pred_lr),
                        evaluate("Random Forest", y_test, y_pred_rf)])
print(results)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.scatter(y_test, y_pred_lr, s=5); plt.title("Linear Regression"); plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.subplot(1, 2, 2); plt.scatter(y_test, y_pred_rf, s=5); plt.title("Random Forest"); plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.show()

**Conclusion:** Random Forest gives a higher R2 and lower RMSE than Linear Regression, so it predicts the Uber fare better.